In [91]:
using LowLevelFEM, LinearAlgebra, SparseArrays

In [92]:
openGeometry("boxes.geo")

In [93]:
#openPreProcessor()

In [94]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);
Ured = Field([mat], type=:VectorField, dim=3, fieldName=:u, reducedOrder=true);

In [95]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 4250)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(K, f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

  5.199714 seconds (116.15 k allocations: 1.124 GiB, 3.18% gc time, 1.96% compilation time)


0


## Penalty contact

The contact object contains only the contact geometry and kinematics. The full
contact operator maps the displacement field to the local contact space,

$$
G: V_u \rightarrow V_c ,
$$

while $P_a$ selects the currently active contact nodes,

$$
G_a=P_aG.
$$

The penalty surface operator is assembled once with the ordinary LLFEM weak-form
machinery and then restricted to the slave surface. In 2D the contact-space
ordering is $(n,t)$. For frictionless contact $c_t=0$.


In [96]:

r = nodePositionVector(U)

C = contact(
    u,
    master="master",
    slave="slave",
    topology_tol=0.01
)

cn = 1e7
ct = 0.0

Dc = [cn 0.0 0.0
      0.0 ct 0.0
      0.0 0.0 ct]

@time C0 = ∫(U ⋅ Dc ⋅ U, Γ="slave")
@time Cc = subSystemMatrix(C0; onPhysicalGroup="slave");


  0.023412 seconds (290.96 k allocations: 29.093 MiB)
  0.009980 seconds (89 allocations: 642.273 KiB)


In [97]:
function lumpContactMatrix(C::SystemMatrix, pdim::Int)
    A = C.A
    d = diag(A)

    for k in 1:pdim
        idx = k:pdim:size(A, 1)

        m = sum(A[idx, idx])   # total weight of this component
        s = sum(d[idx])        # diagonal weight

        abs(s) > eps() && (d[idx] .*= m / s)
    end

    return SystemMatrix(
        spdiagm(0 => d),
        C.model,
        C.test_model,
        C.problems,
        C.offsets
    )
end

#Cc_consistent = Cc
#Cc = lumpContactMatrix(Cc, U.pdim)

lumpContactMatrix (generic function with 1 method)

In [98]:
TT, RR = reductionMatrices(Ured)

(sparse([3289, 3721, 3724, 3727, 3736, 4024, 4042, 34576, 34582, 34585  …  21590, 32000, 32012, 1785, 1842, 1845, 21585, 21591, 32001, 32013], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  11960, 11960, 11960, 11961, 11961, 11961, 11961, 11961, 11961, 11961], [1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5  …  0.5, 0.5, 0.5, 1.0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5], 78834, 11961), sparse([5956, 5957, 5958, 5959, 5960, 5961, 5977, 5978, 5979, 5980  …  11049, 6982, 6983, 6984, 10354, 10355, 10356, 6283, 6284, 6285], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  57651, 57652, 57653, 57654, 57655, 57656, 57657, 57658, 57659, 57660], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 11961, 78834))


At a fixed contact geometry the active penalty contribution is

$$
d_a=P_a d,\qquad
C_a=P_a C_c P_a^T,\qquad
G_a=P_aG,
$$

$$
r_c=G_a^T C_a d_a,\qquad
K_c=G_a^T C_aG_a.
$$

Since $d=G(r+u)$, freezing the current geometry gives the Newton equation

$$
(K+K_c)u_{\mathrm{new}}=f-K_cr.
$$

Thus the linear correction can still be solved with the ordinary `solveField`
function. A line search is retained because the projection and active set may
change during the iteration.


In [99]:

support = [bc_bottom, bc_top]

support_increment = [
    BoundaryCondition("bottom", ux=0, uy=0, uz=0),
    BoundaryCondition("top",    ux=0, uy=0, uz=0)
]

free = freeDoFs(U, support)

# Only used to compare nonlinear residuals on unconstrained DoFs.
freeNorm(v::VectorField) = LinearAlgebra.norm(elementsToNodes(v).a[free, 1])

u_it = copy(u)

old_tags = copy(C.master_element_tags)
old_G = copy(C.G)

Δu = copy(u) * 0

for iter in 1:1#30

    updateContact!(C, u_it)
    #C = contact(u_it, master="master", slave="slave", topology_tol=0.01)

    nchanged = count(old_tags .!= C.master_element_tags)

    dG = norm(C.G - old_G) /
         max(norm(old_G), eps())

    old_tags = copy(C.master_element_tags)
    old_G = copy(C.G)

    # Active contact algebra
    Ga = C.Pa * C.G
    Ca = C.Pa * Cc * C.Pa'
    da = C.Pa * C.d

    # Contact residual and frozen-geometry tangent
    rc = Ga' * (Ca * da)
    Kc = Ga' * Ca * Ga

    # Total residual
    R = K * u_it - f + rc
    R0 = freeNorm(R)

    # Full frozen-geometry Newton step:
    # (K + Kc) u_new = f - Kc r
    #=
    fill!(Δu.a, 0.0)
    Aff = (K + Kc).A[free, free]
    bf = DoFs(-R)[free]
    F = ldlt(Symmetric(Aff))
    Δu.a[free] = F \ bf
    =#
    
    Δu = solveField(
        K + Kc,
        -R,
        support=support_increment
    )
    
    Δu.a[:, 1] .= TT * (RR * Δu.a[:, 1])

    # Line search because G, projection and the active set change with u.
    α = 1.0

    u_trial = copy(u_it)
    Rtrial = R

    while α > 1e-6

        u_trial = u_it + α * Δu

        updateContact!(C, u_trial)

        Ga_trial = C.Pa * C.G
        Ca_trial = C.Pa * Cc * C.Pa'
        da_trial = C.Pa * C.d

        rc_trial = Ga_trial' * (Ca_trial * da_trial)

        Rtrial = K * u_trial - f + rc_trial

        freeNorm(Rtrial) < R0 && break

        α *= 0.5
    end

    u_it = u_trial

    err = freeNorm(α * Δu) /
          max(freeNorm(u_it), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(C.active),
        ", master changes = ", nchanged,
        ", dG = ", dG,
        ", min gap = ", minimum(C.gap_values),
        ", error = ", err,
        ", |R| = ", freeNorm(Rtrial)
    )

    err < 1e-8 && break
end

u = u_it

# Synchronize the stored contact state with the converged field.
updateContact!(C, u);


iter = 1, α = 1.0, active = 281, master changes = 0, dG = 4.116935760608053e-12, min gap = -0.0005980956955236197, error = 0.04772917035589379, |R| = 6632.691695284839


In [100]:

showDoFResults(u, name="u", factor=1, visible=true)


1


## Contact fields

`C.d` is a reduced `ContactVector`. Mapping it back to the displacement mesh
makes the local contact components available through the ordinary field API.

For the current closest-point geometry, `D[1]` is the normal gap. The tangential
component of the current position difference is approximately zero by
construction; tangential slip will later be accumulated from displacement
increments/history.


In [101]:

Dn = VectorField(C.d)

gap = Dn[1]

# Active normal gap, expanded back to the full contact space.
da_full = C.Pa' * (C.Pa * C.d)
Da = VectorField(da_full)

# Pointwise penalty traction (positive in compression).
pressure = -cn * Da[1]

gap

showElementResults(nodesToElements(pressure, onPhysicalGroup="slave"), name="p")
showElementResults(nodesToElements(gap, onPhysicalGroup="slave"), name="gap")
showElementResults(C.gap, name="orig gap")


4

In [102]:
@showdef C

struct Contact:
  master::String
  slave::String
  U::Problem
  multiplier::Union{Nothing, Problem}
  displacement::VectorField
  slave_nodes::Vector{Int64}
  master_element_tags::Vector{Int64}
  master_local_coordinates::Vector{Vector{Float64}}
  master_points::Matrix{Float64}
  gap::ScalarField
  gap_values::Vector{Float64}
  d::ContactVector
  G::SystemMatrix
  Pa::SystemMatrix
  E::Union{Nothing, SystemMatrix}
  n::VectorField
  t1::VectorField
  t2::Union{Nothing, VectorField}
  active::BitVector
  multiplier_dofs::Vector{Int64}
  options::NamedTuple


In [103]:
showElementResults(nodesToElements(Dn, onPhysicalGroup="slave"))
showElementResults(nodesToElements(Da, onPhysicalGroup="slave"))

6

In [104]:

# Example postprocessing:
# showElementResults(nodesToElements(gap), name="gap", visible=true)
# showElementResults(nodesToElements(pressure), name="pressure", visible=true)

openPostProcessor()
